# NOX vs Temperature Correlation

This notebook loads all CSV files in `co_and_nox_emission_data_set` and computes correlation matrices between **NOX** and temperature-related columns (AT, TIT, TAT).

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("co_and_nox_emission_data_set")
files = sorted(DATA_DIR.glob("*.csv"))

if not files:
    raise FileNotFoundError(f"No CSV files found in {DATA_DIR.resolve()}")

# Load and concatenate all yearly files
frames = []
for file in files:
    df = pd.read_csv(file)
    df["_source_file"] = file.name
    frames.append(df)

all_data = pd.concat(frames, ignore_index=True)

# Identify temperature columns (expected in dataset)
expected_temp_cols = ["AT", "TIT", "TAT"]
temp_cols = [col for col in expected_temp_cols if col in all_data.columns]

if "NOX" not in all_data.columns:
    raise KeyError("Column 'NOX' not found in dataset.")

# Overall correlation matrix for NOX and temperatures
corr_overall = all_data[["NOX"] + temp_cols].corr()

# Per-file correlation (NOX vs temperatures)
per_file_corr = (
    all_data[["NOX", *temp_cols, "_source_file"]]
    .groupby("_source_file")
    .corr()
    .loc[(slice(None), "NOX"), temp_cols]
    .droplevel(1)
    .sort_index()
)

corr_overall, per_file_corr

(          NOX        AT       TIT       TAT
 NOX  1.000000 -0.558174 -0.213865 -0.092791
 AT  -0.558174  1.000000  0.183706  0.281869
 TIT -0.213865  0.183706  1.000000 -0.380862
 TAT -0.092791  0.281869 -0.380862  1.000000,
                     AT       TIT       TAT
 _source_file                              
 gt_2011.csv  -0.651208 -0.231984  0.087566
 gt_2012.csv  -0.566584 -0.226817 -0.093118
 gt_2013.csv  -0.581687 -0.122998 -0.179357
 gt_2014.csv  -0.656706 -0.236626 -0.227395
 gt_2015.csv  -0.593580 -0.520081  0.054455)